Broad cross-asset example list you can try in `assets` configs:

- iShares Core S&P 500 (`CSSPX.MI`)
- iShares Core MSCI World (`SWDA.MI`)
- iShares Core MSCI Emerging Markets IMI (`EIMI.MI`)
- iShares Nasdaq 100 (`CSNDX.MI`)
- iShares MSCI ACWI (`IUSQ.MI`)
- Vanguard FTSE All-World (`VWCE.MI`)
- iShares Core DAX (`EXS1.MI`)
- Lyxor Core STOXX Europe 600 (DR) (`MEUD.MI`)
- iShares Core MSCI Europe (`SMEA.MI`)
- Xtrackers MSCI USA (`XD9U.MI`)
- Xtrackers MSCI Emerging Markets (`XMME.MI`)
- iShares Core EURO STOXX 50 (`CSSX5E.MI`)
- Real Estate Sector (`XLRE`)
- ETF Bond USA 7 (`GOVT`)
- ETF Bond USA 10 (`TLH`)
- ETF Bond USA 15 (`LTPZ`)
- ETF Bond USA 20 (`XTLT.TO`)
- ETF Bond USA 30 (`UTHY`)
- ETF Inflation Adjusted USA 7 (`TIP`)
- ETF Inflation Adjusted USA 15 (`LTPZ`)
- Bitcoin (`BTC=F`)
- Gold (`GC=F`)
- Silver (`SI=F`)
- Crude Oil (`CL=F`)
- Cash Liquidity (`CSH.PA`)


### Assets Definition

In [1]:
#asset defintion 
from orchestration import *


asset_info = [
        SyntheticLETFAssetConfig(
            id="SWDA_2X",
            underlying_ticker="SWDA.MI",
            leverage=2.0,
            ter=0.0092,
            spread=0.0030,
        ),
        SpotAssetConfig(
            id="GOVIES",
            ticker="GOVT",
        ),
    ]

### Simulation

In [ ]:
import importlib

import data_loader
import orchestration

importlib.reload(data_loader)
importlib.reload(orchestration)
from orchestration import *

# Rebuild assets with refreshed dataclass types after module reload.
normalized_asset_info = []
for asset in asset_info:
    if hasattr(asset, "underlying_ticker"):
        normalized_asset_info.append(
            SyntheticLETFAssetConfig(
                id=asset.id,
                underlying_ticker=asset.underlying_ticker,
                leverage=float(asset.leverage),
                ter=float(asset.ter),
                spread=float(getattr(asset, "spread", 0.0)),
            )
        )
    elif hasattr(asset, "ticker"):
        normalized_asset_info.append(
            SpotAssetConfig(
                id=asset.id,
                ticker=asset.ticker,
            )
        )
    else:
        raise ValueError(f"Unsupported asset entry: {asset}")

asset_info = normalized_asset_info

# Define the simulation configuration 
# with market data, assets, portfolio settings, and Monte Carlo parameters.
config = SimulationConfig(
    market=MarketDataConfig(
        start="2018-04-03",
        end="2025-12-31",
        fred_series="SOFR",
        fred_is_percent=True,
    ),
    assets=asset_info,
    portfolio=PortfolioConfig(
        target_weights={
            "SWDA_2X": 0.80,
            "GOVIES": 0.20,
        },
        initial_capital=100_000.0,
        rebalance_frequency_days=252,
        tolerance_band=0.05,
        capital_gains_tax_rate=0.26,
    ),
    monte_carlo=MonteCarloConfig(
        n_paths=5000,
        horizon_days=2520,
        method="bootstrap",
        distribution="normal",
        student_t_df=6.0,
        seed=42,
    ),
    metrics_ruin_threshold_fraction=0.10,
    use_mean_risk_free_for_metrics=True,
)

result = run_complete_simulation(config)

print("Wealth paths shape:", result.portfolio.wealth_paths.shape)
print("Simulated returns shape:", result.simulated_asset_returns.shape)
print("Metrics summary:")

metrics_summary = result.metrics.summary
print(metrics_summary.to_string(col_space=12, justify="right"))




Base tickers needed for historical data:
{'SWDA.MI', 'GOVT'}


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:218: RuntimeWarning: Local cache miss for symbols: ['GOVT', 'SWDA.MI']. Attempting online providers.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:327: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change()


### Visuals

In [56]:
import importlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import visuals
from orchestration import SpotAssetConfig, SyntheticLETFAssetConfig
from plotly.subplots import make_subplots

importlib.reload(visuals)

from visuals import (
    plot_drawdown_chart,
    plot_spaghetti_paths,
    plot_terminal_wealth_distribution,
)

FONT_STACK = "Satoshi, 'Satoshi Variable', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif"

wealth_paths = result.portfolio.wealth_paths
terminal = wealth_paths[:, -1]

terminal_summary = pd.Series(
    {
        "min": float(np.min(terminal)),
        "p5": float(np.quantile(terminal, 0.05)),
        "median": float(np.median(terminal)),
        "mean": float(np.mean(terminal)),
        "p95": float(np.quantile(terminal, 0.95)),
        "max": float(np.max(terminal)),
    },
    name="TerminalWealth",
)

display(terminal_summary.to_frame())

asset_lines = []
for asset in config.assets:
    weight = float(config.portfolio.target_weights.get(asset.id, 0.0))

    if isinstance(asset, SpotAssetConfig):
        asset_lines.append(f"{asset.ticker} | weight={weight:.1%}")
    elif isinstance(asset, SyntheticLETFAssetConfig):
        asset_lines.append(
            f"{asset.underlying_ticker} {asset.leverage:.1f}x | weight={weight:.1%} | TER={asset.ter:.2%} | spread={asset.spread:.2%}"
        )

assets_subtitle = "<br>".join(asset_lines)

summary_note = (
    f"FRED={config.market.fred_series}"
    "<br>"
    f"{config.monte_carlo.n_paths:,} draws | {config.monte_carlo.horizon_days} horizon days"
    "<br>"
    f"{config.monte_carlo.method} Method<br>Distribution={config.monte_carlo.distribution}<br>Student_t_df={config.monte_carlo.student_t_df}"
    "<br>"
    f"Initial capital={config.portfolio.initial_capital:,.2f}<br>Rebalancing Frequency={config.portfolio.rebalance_frequency_days} days"
    "<br>"
    f"Tolerance band={config.portfolio.tolerance_band:.2%}<br>Capital Gain Tax={config.portfolio.capital_gains_tax_rate:.2%}"
)

fig_spaghetti = plot_spaghetti_paths(
    wealth_paths=wealth_paths,
    n_sample=100,
    seed=42,
    normalize_to_1=False,
    title="Monte Carlo Spaghetti",
    subtitle=assets_subtitle,
    bottom_note=summary_note,
    subtitle_align="left",
    bottom_note_align="left",
    bottom_note_x=0.65,
    bottom_note_y=-0.10,
    bottom_note_box=True,
    backend="plotly",
    width=800,
    height=600,
)

fig_spaghetti.update_xaxes(title_text="")

fig_terminal = plot_terminal_wealth_distribution(
    wealth_paths=wealth_paths,
    bins=60,
    title="Distribution of Terminal Wealth",
    backend="plotly",
    width=800,
    height=600,
)

terminal_footnote = (
    f"min={terminal_summary['min']:,.0f} | p5={terminal_summary['p5']:,.0f} | "
    f"median={terminal_summary['median']:,.0f} | mean={terminal_summary['mean']:,.0f} | "
    f"p95={terminal_summary['p95']:,.0f} | max={terminal_summary['max']:,.0f}"
)

fig_combined = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.28,
    row_heights=[0.5, 0.5],
)

for tr in fig_spaghetti.data:
    fig_combined.add_trace(tr, row=1, col=1)

for tr in fig_terminal.data:
    fig_combined.add_trace(tr, row=2, col=1)

fig_combined.update_layout(
    width=900,
    height=1050,
    margin={"t": 110, "b": 100, "l": 70, "r": 30},
    showlegend=fig_spaghetti.layout.showlegend if fig_spaghetti.layout.showlegend is not None else False,
)

fig_combined.update_xaxes(title_text="", row=1, col=1)
fig_combined.update_yaxes(
    title_text=fig_spaghetti.layout.yaxis.title.text if fig_spaghetti.layout.yaxis.title is not None else "Portfolio Value",
    row=1,
    col=1,
    showgrid=fig_spaghetti.layout.yaxis.showgrid if fig_spaghetti.layout.yaxis.showgrid is not None else True,
    gridcolor=fig_spaghetti.layout.yaxis.gridcolor if fig_spaghetti.layout.yaxis.gridcolor is not None else None,
)

fig_combined.update_xaxes(
    title_text=fig_terminal.layout.xaxis.title.text if fig_terminal.layout.xaxis.title is not None else "Terminal Wealth",
    row=2,
    col=1,
    showgrid=fig_terminal.layout.xaxis.showgrid if fig_terminal.layout.xaxis.showgrid is not None else True,
    gridcolor=fig_terminal.layout.xaxis.gridcolor if fig_terminal.layout.xaxis.gridcolor is not None else None,
)
fig_combined.update_yaxes(
    title_text=fig_terminal.layout.yaxis.title.text if fig_terminal.layout.yaxis.title is not None else "Frequency",
    row=2,
    col=1,
    showgrid=fig_terminal.layout.yaxis.showgrid if fig_terminal.layout.yaxis.showgrid is not None else True,
    gridcolor=fig_terminal.layout.yaxis.gridcolor if fig_terminal.layout.yaxis.gridcolor is not None else None,
)

fig_combined.add_annotation(
    x=0.0,
    y=1.12,
    xref="paper",
    yref="paper",
    text="Monte Carlo Spaghetti",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font={"size": 20, "color": "#0f172a"},
)

fig_combined.add_annotation(
    x=0.0,
    y=1.07,
    xref="paper",
    yref="paper",
    text=assets_subtitle,
    showarrow=False,
    xanchor="left",
    yanchor="top",
    align="left",
    font={"size": 12, "color": "#334155"},
)

summary_box_y = 0.59
metrics_box_y = 0.59

fig_combined.add_annotation(
    x=0.65,
    y=summary_box_y,
    xref="paper",
    yref="paper",
    text=summary_note,
    showarrow=False,
    xanchor="left",
    yanchor="top",
    align="left",
    font={"size": 11, "color": "#334155"},
    bgcolor="rgba(255, 255, 255, 0.82)",
    bordercolor="#cbd5e1",
    borderwidth=1,
)

metrics_summary = result.metrics.summary
if isinstance(metrics_summary, pd.DataFrame):
    if metrics_summary.shape[1] == 1:
        metrics_series = metrics_summary.iloc[:, 0]
    else:
        metrics_series = metrics_summary.mean(axis=1)
else:
    metrics_series = pd.Series(metrics_summary)

def _fmt_metric_value(v: object) -> str:
    if isinstance(v, (float, np.floating)):
        if abs(v) >= 1000:
            return f"{v:,.2f}"
        return f"{v:.4f}"
    return str(v)

metrics_table_df = pd.DataFrame(
    {
        "Metric": [str(idx) for idx in metrics_series.index],
        "Value": [_fmt_metric_value(v) for v in metrics_series.values],
    }
)

n_metrics = len(metrics_table_df)
n_rows = max(1, int(np.ceil(n_metrics / 2)))
left_metrics = metrics_table_df.iloc[:n_rows].reset_index(drop=True)
right_metrics = metrics_table_df.iloc[n_rows:].reset_index(drop=True)
if len(right_metrics) < n_rows:
    right_metrics = pd.concat(
        [
            right_metrics,
            pd.DataFrame(
                {
                    "Metric": [""] * (n_rows - len(right_metrics)),
                    "Value": [""] * (n_rows - len(right_metrics)),
                }
            ),
        ],
        ignore_index=True,
    )

table_top = metrics_box_y
table_height = 0.13
table_left = 0.00
table_width = 0.58

row_fill = ["#ffffff" if i % 2 == 0 else "#f8fafc" for i in range(n_rows)]

fig_combined.add_trace(
    go.Table(
        columnwidth=[0.34, 0.16, 0.34, 0.16],
        header={
            "values": ["<b>Metric</b>", "<b>Value</b>", "<b>Metric</b>", "<b>Value</b>"],
            "align": ["left", "right", "left", "right"],
            "fill_color": "#e2e8f0",
            "font": {"family": "Satoshi", "size": 11, "color": "#0f172a"},
            "line_color": "#cbd5e1",
            "height": 24,
        },
        cells={
            "values": [
                left_metrics["Metric"],
                left_metrics["Value"],
                right_metrics["Metric"],
                right_metrics["Value"],
            ],
            "align": ["left", "right", "left", "right"],
            "fill_color": [row_fill, row_fill, row_fill, row_fill],
            "font": {"family": "Satoshi", "size": 10, "color": "#0f172a"},
            "line_color": "#e2e8f0",
            "height": 21,
        },
        domain={
            "x": [table_left, table_left + table_width],
            "y": [table_top - table_height, table_top],
        },
    )
)

fig_combined.add_annotation(
    x=0.0,
    y=0.40,
    xref="paper",
    yref="paper",
    text="Distribution of Terminal Wealth",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font={"size": 18, "color": "#0f172a"},
)

fig_combined.add_annotation(
    x=0.1,
    y=-0.05,
    xref="paper",
    yref="paper",
    text=terminal_footnote,
    showarrow=False,
    align="left",
    xanchor="left",
    yanchor="top",
    font={"size": 11, "color": "#334155"},
)

fig_drawdown = plot_drawdown_chart(
    wealth_paths=wealth_paths,
    drawdowns=result.metrics.drawdowns,
    title="Drawdown Chart: Median vs Worst Case",
    backend="plotly",
    width=1200,
    height=720,
)

for fig in (fig_spaghetti, fig_terminal, fig_combined, fig_drawdown):
    fig.update_layout(
        font={"family": FONT_STACK, "size": 12, "color": "#0f172a"},
        title_font={"family": "Satoshi"},
    )
    fig.update_xaxes(tickfont={"family": "Satoshi"}, title_font={"family": "Satoshi"})
    fig.update_yaxes(tickfont={"family": "Satoshi"}, title_font={"family": "Satoshi"})

if fig_combined.layout.annotations:
    for ann in fig_combined.layout.annotations:
        existing_font = ann.font.to_plotly_json() if ann.font else {}
        ann.font = {**existing_font, "family": "Satoshi"}

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M")
output_path = output_dir / f"mc-simulation-{timestamp}.png"
fig_combined.write_image(output_path, format="png", width=900, height=1050, scale=2)
print(f"Saved final figure to: {output_path}")

fig_combined.show()
# fig_drawdown.show()

,TerminalWealth
min,2.111329e+04
p5,1.195089e+05
median,4.511531e+05
mean,6.244254e+05
p95,1.697322e+06
max,1.439998e+07


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
